# 第78章 瀑布图（Waterfall）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 15 / 18 步：表达层级、流程、贡献与地域**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 漏斗图（px.funnel）  →  **本章任务：** 瀑布图（Waterfall）  →  **下一步：** 时间线与甘特图（px.timeline）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

看月度经营数据时，我们常想知道「这个月利润为什么涨了或跌了」——是销量上来了，还是某笔费用超支了？瀑布图（Waterfall）正好擅长回答这类问题：它把一条「起点」逐步拆成若干正负增减段，再汇到「终点」，每个因素贡献了多少一目了然。


## 本章目标

学完本章，你将能够：

- **理解**：理解「瀑布图（Waterfall）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「瀑布图（Waterfall）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「瀑布图（Waterfall）」并读出其中的结论。


## 适用场景

**背景引入**：看月度经营数据时，我们常想知道「这个月利润为什么涨了或跌了」——是销量上来了，还是某笔费用超支了？瀑布图（Waterfall）正好擅长回答这类问题：它把一条「起点」逐步拆成若干正负增减段，再汇到「终点」，每个因素贡献了多少一目了然。很多看板和报表都用它来讲利润、预算或库存的增减故事，这一章我们就动手把它做出来、看懂它。（可以把它想成记账本的余额变化：月初是起点，中间一笔笔“+”和“-”追加或扣减，月底落到终点；measure 就是告诉每一笔是“起点”“增减”还是“总计”，瀑布图把它们按顺序连成一条余额曲线。）

分析利润、预算、用户或库存的增减贡献。


## 数据结构

有序贡献项、正负变化值以及起止汇总项。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 textposition="outside" 改为 "inside"，观察数值标签位置对可读性的影响
2. 修改 increasing 和 decreasing 的 marker color，对比不同正负配色方案的区分度
3. 将某个 measure 项从 "relative" 改为 "total"，说明汇总项对累计读取的作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `go.Figure()`、`go.Waterfall()`、`fig.update_layout()`、`fig.show()` | 分析利润、预算、用户或库存的增减贡献。 | 贡献项不满足可加性 |
| 进阶变体 | `go.Figure()`、`go.Waterfall()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 起点和终点口径不同 |
| 关键参数 | `measure` | relative/total/absolute | 贡献项不满足可加性 |
| 关键参数 | `connector` | 连接线 | 起点和终点口径不同 |
| 关键参数 | `increasing/decreasing` | 颜色 | 项目过多导致难读 |
| 关键参数 | `text` | 标签 | 贡献项不满足可加性 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-78 -->
### 数学推导｜瀑布图的逐步守恒

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜每一步更新一次余额。** $V_k=V_{k-1}+\Delta_k$。

**第 2 步｜连续代入前一步。** $V_2=V_0+\Delta_1+\Delta_2$，继续展开可得

$$
V_k=V_0+\sum_{i=1}^{k}\Delta_i
$$

**第 3 步｜用终点做校验。** 若报表给出终值 $V_{reported}$，应检查 $V_0+\sum_i\Delta_i-V_{reported}=0$ 或只存在可解释的舍入差。

**把上面的关系收束为本章计算式：**

$$
V_k=V_0+\sum_{i=1}^{k}\Delta_i
$$

**符号解释：** $V_0$ 是起点，$\Delta_i$ 是第 $i$ 项正负变化，$V_k$ 是累计结果。

**代码对应：** 逐项计算变化并检查最终累计值是否等于报表终值。

**使用边界：** 变化项必须互斥且口径一致，否则“贡献”会被重复计算。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = go.Figure(
    go.Waterfall(
        name="利润变化",
        orientation="v",
        measure=[
            "absolute",
            "relative",
            "relative",
            "relative",
            "relative",
            "total",
        ],
        x=["上期利润", "销售增长", "提价", "营销费用", "物流费用", "本期利润"],
        y=[120, 48, 22, -18, -12, 0],
        connector={"line": {"color": "#9aa0a6"}},
        textposition="outside",
    )
)
fig.update_layout(
    title="利润变化贡献", yaxis_title="利润（万元）", showlegend=False
)
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


#### 练一练：观察一个参数对瀑布图的改变

上面的基础瀑布图默认把数值标签放在柱子**外侧**（`textposition='outside'`），`y` 中每个段对应一个贡献项。请做两处改动再重跑：① 把 `textposition` 改成 `'inside'`，观察标签落到柱内对可读性的影响；② 把 `y` 里某个正数段（例如 `48`）改成别的值（比如 `30`），看看对应的「销售增长」段在图上怎么变。

请把你改动的参数和观察到现象填进下方练习代码的变量里。


In [ ]:
try:
    # 请在下方填写代码：复制上面的 go.Figure 瀑布图，
    # ① 把 textposition 由 'outside' 改为 'inside'；
    # ② 把 y 中某个 relative 段的值改掉（例如把 48 改成 30）。
    # 记录：我改了什么？预期是什么？实际观察到什么？
    your_change = "待填写：textposition 或 y 值"
    your_note = "运行后填写你的观察"

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = go.Figure(
    go.Waterfall(
        measure=["absolute", "relative", "relative", "relative", "total"],
        x=["期初用户", "新增用户", "召回用户", "流失用户", "期末用户"],
        y=[8200, 1400, 360, -920, 0],
        increasing={"marker": {"color": "#188038"}},
        decreasing={"marker": {"color": "#d93025"}},
        totals={"marker": {"color": "#1a73e8"}},
        textposition="outside",
    )
)
fig.update_layout(title="月度用户规模变化", yaxis_title="用户数")
fig.show()


## 参数说明

- measure：relative/total/absolute
- connector：连接线
- increasing/decreasing：颜色
- text：标签


## 结果解读

从左到右累计读取，区分正贡献、负贡献和最终总计。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 贡献项不满足可加性
- 起点和终点口径不同
- 项目过多导致难读


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：换一组经营数据，练习瀑布图的增减项设计
    # 【目标】换一组成本数据，练习设计 measure 的 absolute/relative/total。
    import plotly.graph_objects as go

    # 起点示例(已可运行)：换一张「成本构成」图，观察增减项设置。
    fig = go.Figure(
        go.Waterfall(
            name="成本构成",
            orientation="v",
            measure=[
                "absolute",
                "relative",
                "relative",
                "relative",
                "relative",
                "total",
            ],
            x=["总成本", "原材料", "人工", "物流", "营销", "净成本"],
            y=[200, 90, 40, 25, 15, 0],
            connector={"line": {"color": "#9aa0a6"}},
            textposition="outside",
        )
    )
    fig.update_layout(
        title="成本构成贡献", yaxis_title="成本（万元）", showlegend=False
    )
    fig.show()

    # ---- 反思记录：增减项中哪些是增加、哪些是减少 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用瀑布图解释一个起点如何经过多个增减项到达终点。


### 你已经掌握

- 判断瀑布图（Waterfall）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `measure` | relative/total/absolute |
| `connector` | 连接线 |
| `increasing/decreasing` | 颜色 |
| `text` | 标签 |


### 需要注意

- 贡献项不满足可加性
- 起点和终点口径不同
- 项目过多导致难读


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
fig = go.Figure(
    go.Waterfall(
        name="利润变化",
        orientation="v",
        measure=[
            "absolute",
            "relative",
            "relative",
            "relative",
            "relative",
            "total",
        ],
        x=["上期利润", "销售增长", "提价", "营销费用", "物流费用", "本期利润"],
        y=[120, 30, 22, -18, -12, 0],
        textposition="inside",
    )
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
fig = go.Figure(
    go.Waterfall(
        measure=["absolute", "relative", "relative", "relative", "total"],
        x=["预算", "人力节省", "工具采购", "外包费用", "最终结余"],
        y=[100, 12, -18, -24, 0],
        textposition="outside",
    )
)
fig.update_layout(title="项目预算变化", yaxis_title="金额（万元）")
fig.show()
